# Nakhlah: train the YOLO26s LEAF DETECTOR (tier 1) for the Jetson
**What this notebook does:** trains **YOLO26s** to find **leaves** in a whole photo (one class: `leaf`). The EdgeNeXt model from the final notebook then decides the disease of each leaf (tier 2).

| Setting | Value |
|---|---|
| Model | Ultralytics **YOLO26s** (COCO pretrained, ~10M numbers), 1 class: `leaf` |
| Boxes | Your Roboflow boxes. All 4 disease labels are merged into `leaf`; insect boxes are left out |
| Split | **The same frozen split** as every run (Run 5 `split.csv` + `roles.csv`, and the final notebook's `qassim_split.csv`). The notebook **stops** if it changed |
| Training | 85% of Al Ammariyah + Al Thumamah + Madinah, folds 1–4, + Qassim folds 1–4 |
| Validation (picks the best epoch) | Fold 0 of the same photos |
| Locked test (used **once**) | The same locked 15% as the EdgeNeXt final model (270 photos) |
| Recipe | 640 px, up to 100 epochs (stops early after 30 with no gain), batch 16, seed 42, Ultralytics default augmentation |

**Results:**
1. **Detector on the locked test:** precision, recall, mAP50, mAP50-95.
2. **Full pipeline on the locked test:** YOLO26s finds the leaves → each box cropped like in training (10% padding, 224 px) → EdgeNeXt classifies → the photo's answer = majority vote. Compared with the same pipeline on the hand-drawn boxes (whole-photo F1 0.855).

**Saved to `MyDrive/Nakhlah_YOLO26s_leaf_detector`:** `nakhlah_yolo26s.pt` (Jetson), `nakhlah_yolo26s.onnx`, `nakhlah_yolo.json`, `jetson_test_photos.zip` (20 locked-test photos), training curves and result CSVs.

**How to run:** Runtime → Change runtime type → **T4 GPU**. Paste your Roboflow key in cell 1. **Run all.** About 1–1.5 hours. The final EdgeNeXt folder (`Nakhlah_EdgeNeXt_FINAL_model`) must be in your Drive (cell 10 uses it).

**Check:** cells 3, 5 and 6 must print **"✅ Same split as the frozen one"**.

In [ ]:
# ===== CELL 1: settings + connect Drive  (YOLO26s leaf detector) =====
!pip install -q "ultralytics>=8.4.0" roboflow pillow-heif timm onnx onnxslim
from google.colab import drive
drive.mount("/content/drive")
import os

ROBOFLOW_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY_HERE"
WORKSPACE = "lubn0-icloud-com"
SITES = {"Ammariyah": ("alamariah", 1),
         "Thumamah": ("althumamah", 1)}     # same versions as before, so the split matches
OLD = ("combined-dataset-for-naklah", 3)      # the old dataset: Madinah + Qassim photos
CLASS_MAP = {}                                # only if cell 2 asks: {"your class name": "old class name"}

SAVE = "/content/drive/MyDrive/Nakhlah_YOLO26s_leaf_detector"      # results of THIS notebook (new folder)
FINAL = "/content/drive/MyDrive/Nakhlah_EdgeNeXt_FINAL_model"      # the final EdgeNeXt model + its frozen Qassim split
RUN5 = "/content/drive/MyDrive/Nakhlah_ATM_combined_5folds"        # Run 5 folder: its frozen split is reused
FREEZE_DIR = RUN5 if os.path.exists(RUN5 + "/roles.csv") else SAVE
SPLIT_FILE = FREEZE_DIR + "/split.csv"
N_FOLDS, SEED = 5, 42
TEST_FRACTION = 0.15
BLOCK = 10
SIZE = 224                             # EdgeNeXt crop size (same as training)
PAD = 0.10                             # 10% extra around each box (same as training)
MIN_PX = 20                            # boxes smaller than 20 px are not classified (same as training)

# YOLO settings
YOLO_MODEL = "yolo26s.pt"              # COCO-pretrained YOLO26 small
IMGSZ = 640
EPOCHS, PATIENCE, YBATCH = 100, 30, 16
VAL_FOLD = 0                           # fold 0 = validation (picks the best epoch); folds 1-4 = training
CONF = 0.25                            # a box counts as a leaf if YOLO is at least 25% sure
GT_PIPELINE = dict(accuracy=0.859, macro_F1=0.855)   # final EdgeNeXt, whole photo, hand-drawn boxes (locked test)
print("Settings ready | split taken from:", FREEZE_DIR)

Mounted at /content/drive
Settings ready | split taken from: /content/drive/MyDrive/Nakhlah_YOLO26s_leaf_detector


In [ ]:
# ===== CELL 2: download your two projects (+ the old dataset) and label every photo =====
import os, glob, re, shutil, hashlib, random, collections
import numpy as np, pandas as pd, yaml
from roboflow import Roboflow
from pillow_heif import register_heif_opener
register_heif_opener()
rf = Roboflow(api_key=ROBOFLOW_KEY)
ws = rf.workspace(WORKSPACE)


def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def load(project, version, folder):
    shutil.rmtree(folder, ignore_errors=True)
    ws.project(project).version(version).download("yolov12", location=folder)
    names = yaml.safe_load(open(folder + "/data.yaml"))["names"]
    names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
    rows, empty = [], 0
    for img in glob.glob(folder + "/*/images/*"):
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        cls = [int(float(l.split()[0])) for l in open(lab) if l.strip()] if os.path.exists(lab) else []
        if not cls:
            empty += 1
            continue
        base = os.path.basename(img).split(".rf.")[0]
        rows.append(dict(img=img, name=base, cls=names[collections.Counter(cls).most_common(1)[0][0]]))
    print(f"{project} v{version}: {len(rows)} photos with boxes, {empty} without boxes (skipped), classes = {names}")
    return pd.DataFrame(rows)


parts = []
for site, (project, version) in SITES.items():
    d = load(project, version, f"/content/ds_{site}")
    d["site"] = site
    num = d.name.str.extract(r"(\d+)")[0]
    d["group"] = [f"{site}_{int(n) // BLOCK}" if isinstance(n, str) else f"{site}_{b}" for n, b in zip(num, d.name)]
    parts.append(d)
df = pd.concat(parts, ignore_index=True)

# one class list for both sites (same class written differently, e.g. "Healthy" / "healthy", is merged)
df["key"] = df.cls.map(norm)
NAMES = sorted(df.groupby("key").cls.first())
KEY2NAME = {norm(n): n for n in NAMES}
df["cls"] = df.key.map(KEY2NAME)
C = {n: i for i, n in enumerate(NAMES)}

# exact duplicate photos: keep one copy (same photo with different classes: removed)
df["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df.img]
bad = df.groupby("md5").cls.nunique()
df = df[~df.md5.isin(bad[bad > 1].index)].drop_duplicates("md5").reset_index(drop=True)

print("\nPhotos per class and site:")
print(pd.crosstab(df.cls, df.site, margins=True).to_string())
print(f"\nLeaf groups: {df.group.nunique()} (blocks of {BLOCK} neighbouring photo numbers)")
print("Example names:", list(df.name.head(5)))

# the old dataset: only its Madinah photos are used, for TRAINING (Qassim photos are dropped in cell 5)
old = load(OLD[0], OLD[1], "/content/ds_old")
old["cls"] = old.cls.map(lambda c: CLASS_MAP.get(c, c))
old["key"] = old.cls.map(norm)
missing = sorted(set(old.key) - set(KEY2NAME))
old = old[old.key.isin(KEY2NAME)]
oldtest = [(p, C[KEY2NAME[k]]) for p, k in zip(old.img, old.key)]
print(f"\nOld dataset photos usable as an other-place test: {len(oldtest)}")
if missing:
    print("⚠️ These old class names have no match in your classes, so those photos are skipped:", missing,
          "\n   If they mean the same thing, add them to CLASS_MAP in cell 1, e.g. {'nutrientDeficiency': 'Deficiency'}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/ds_Ammariyah in yolov12:: 100%|██████████| 2032/2032 [00:01<00:00, 1872.82it/s]


alamariah v1: 1014 photos with boxes, 1 without boxes (skipped), classes = ['fungalSpotting', 'fungalWiltBlight', 'healthy', 'nutrientDeficiency']
loading Roboflow project...



Extracting Dataset Version Zip to /content/ds_Thumamah in yolov12:: 100%|██████████| 2090/2090 [00:00<00:00, 3067.14it/s]


althumamah v1: 1027 photos with boxes, 17 without boxes (skipped), classes = ['fungalSpotting', 'fungalWiltBlight', 'healthy', 'nutrientDeficiency']

Photos per class and site:
site                Ammariyah  Thumamah   All
cls                                          
fungalSpotting            385       159   544
fungalWiltBlight          300       164   464
healthy                   140       553   693
nutrientDeficiency        189       151   340
All                      1014      1027  2041

Leaf groups: 226 (blocks of 10 neighbouring photo numbers)
Example names: ['IMG_9340_JPG', 'IMG_1549_JPG', 'IMG_8977_JPG', 'IMG_2379_HEIC', '20261011_115834_jpg']
loading Roboflow project...



Extracting Dataset Version Zip to /content/ds_old in yolov12:: 100%|██████████| 1243/1243 [00:00<00:00, 5736.15it/s]

combined-dataset-for-naklah v3: 620 photos with boxes, 0 without boxes (skipped), classes = ['fungalSpotting', 'fungalWiltBlight', 'healthy', 'insectInfestation', 'nutrientDeficiency']

Old dataset photos usable as an other-place test: 565
⚠️ These old class names have no match in your classes, so those photos are skipped: ['insectinfestation'] 
   If they mean the same thing, add them to CLASS_MAP in cell 1, e.g. {'nutrientDeficiency': 'Deficiency'}


In [ ]:
# ===== CELL 3: SPLIT Al Ammariyah + Al Thumamah: locked TEST (15%) + 5 leaf-grouped folds, then FREEZE it =====
# The same split as Run 5 (checked against Run 5's saved split). Madinah is split the same way in cell 5.
from sklearn.model_selection import StratifiedGroupKFold
os.makedirs(SAVE, exist_ok=True)
sg = StratifiedGroupKFold(n_splits=round(1 / TEST_FRACTION), shuffle=True, random_state=SEED)
strat = df.site + "_" + df.cls                       # keep both sites and all classes spread evenly
p_idx, t_idx = next(sg.split(df, strat, df.group))
df["part"] = "pool"
df.loc[t_idx, "part"] = "locked_test"
df["fold"] = -1
pool = df[df.part == "pool"]
sg = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, v) in enumerate(sg.split(pool, strat[pool.index], pool.group)):
    df.loc[pool.index[v], "fold"] = k

assert set(df[df.part == "pool"].group).isdisjoint(df[df.part == "locked_test"].group), "LEAK: group in pool and test"
assert (df[df.part == "pool"].groupby("group").fold.nunique() == 1).all(), "LEAK: group split across folds"
print("✅ No leakage: every leaf group is in one place only")

FROZEN = SPLIT_FILE
cols = ["site", "name", "cls", "part", "fold"]
if os.path.exists(FROZEN):
    old_split = pd.read_csv(FROZEN, dtype={"name": str})[cols]
    same = sorted(map(tuple, old_split.values.tolist())) == sorted(map(tuple, df[cols].values.tolist()))
    if not same:
        raise SystemExit("❌ STOP: the split CHANGED (did the Roboflow version change?). Nothing was trained.")
    print("✅ Same split as the frozen one")
else:
    os.makedirs(os.path.dirname(FROZEN), exist_ok=True)
    df[cols].to_csv(FROZEN, index=False)
    print("No frozen split found, so this new one was saved to", FROZEN)

print(f"\nPool (5 folds): {(df.part == 'pool').sum()} | Locked TEST (never trained on): {(df.part == 'locked_test').sum()}")
print(pd.crosstab([df.part, df.fold], df.cls).to_string())

✅ No leakage: every leaf group is in one place only
No frozen split found, so this new one was saved to /content/drive/MyDrive/Nakhlah_YOLO26s_leaf_detector/split.csv

Pool (5 folds): 1847 | Locked TEST (never trained on): 194
cls               fungalSpotting  fungalWiltBlight  healthy  nutrientDeficiency
part        fold                                                               
locked_test -1                51                38       65                  40
pool         0                67                36      128                  42
             1               166               115      142                  68
             2               107                90      112                  61
             3                61               121      103                  77
             4                92                64      143                  52


Cell 4 of the classifier notebooks (the training recipe) is not needed here.

In [ ]:
# ===== CELL 5: ALL 3 PLACES COMBINED. Each place: locked TEST (15%) + 5 folds for training (85%). Then FREEZE. =====
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import GroupKFold


def leaf_of(name):
    """Madinah names look like '12-3': the first number is the leaf, so all photos of leaf 12 stay together."""
    m = re.match(r"^[^0-9]*?(\d+)[._\- ](\d+)", str(name))
    return m.group(1) if m else str(name)


def grouped_folds(d, n, strat):
    """Leaf-grouped, class-balanced folds; falls back to group-only if a class is too small."""
    try:
        return list(StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=SEED).split(d, strat, d.group))
    except ValueError:
        return list(GroupKFold(n_splits=n).split(d, strat, d.group))


# Al Ammariyah + Al Thumamah: the split from cell 3 (pool = training with folds, locked = TEST)
farms = df.copy()
farms["source"] = farms.site
farms["side"] = np.where(farms.part == "pool", "train", "test")
farms.loc[farms.side == "test", "fold"] = -1

# Madinah: exact duplicate photos kept once (a photo saved under two classes is removed),
# then the SAME method: locked TEST (15%) + 5 leaf-grouped folds on the rest
mad = old[~old.name.str.upper().str.startswith("QD")].copy()
mad["cls"] = mad.key.map(KEY2NAME)
mad["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in mad.img]
bad = mad.groupby("md5").cls.nunique()
mad = mad[~mad.md5.isin(bad[bad > 1].index)].drop_duplicates("md5").reset_index(drop=True)
mad["group"] = "Madinah_" + mad.cls + "_" + mad.name.map(leaf_of)
_, t_idx = grouped_folds(mad, round(1 / TEST_FRACTION), mad.cls)[0]
mad["side"] = "train"
mad.loc[t_idx, "side"] = "test"
mad["fold"] = -1
mp = mad[mad.side == "train"]
for k, (_, v) in enumerate(grouped_folds(mp, N_FOLDS, mp.cls)):
    mad.loc[mp.index[v], "fold"] = k
mad["source"] = "Madinah"
assert set(mad[mad.side == "train"].group).isdisjoint(mad[mad.side == "test"].group), "LEAK: Madinah leaf in train and test"
assert (mad[mad.side == "train"].groupby("group").fold.nunique() == 1).all(), "LEAK: a Madinah leaf is split across folds"

cols = ["img", "name", "cls", "fold", "source", "side"]
roles = pd.concat([farms[cols], mad[cols]], ignore_index=True)
roles["fold"] = roles.fold.astype(int)
roles["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in roles.img]
assert not set(roles[roles.side == "train"].md5) & set(roles[roles.side == "test"].md5), "LEAK: same photo in training and test"
assert (roles[roles.side == "train"].fold.between(0, N_FOLDS - 1)).all(), "a training photo has no fold"
assert (roles[roles.side == "test"].fold == -1).all()
print("✅ No leakage: no leaf and no photo is in both training and the locked test; every training photo has one fold")

os.makedirs(SAVE, exist_ok=True)
FROZEN2 = FREEZE_DIR + "/roles.csv"          # Run 5's roles: must match exactly
fcols = ["source", "name", "cls", "side", "fold"]
if os.path.exists(FROZEN2):
    fz = pd.read_csv(FROZEN2, dtype={"name": str})[fcols]
    same = sorted(map(tuple, fz.astype(str).values.tolist())) == sorted(map(tuple, roles[fcols].astype(str).values.tolist()))
    if not same:
        raise SystemExit("❌ STOP: the split CHANGED. Nothing was trained.")
    print("✅ Same split as the frozen one")
else:
    roles[fcols].to_csv(FROZEN2, index=False)
    print("Saved the frozen split to", FROZEN2)
print("\nPhotos: training (5 folds) vs locked test, per place:")
print(pd.crosstab(roles.source, roles.side, margins=True).to_string())
print("\nTraining photos per fold:")
print(pd.crosstab(roles[roles.side == "train"].source, roles[roles.side == "train"].fold).to_string())

✅ No leakage: no leaf and no photo is in both training and the locked test; every training photo has one fold
Saved the frozen split to /content/drive/MyDrive/Nakhlah_YOLO26s_leaf_detector/roles.csv

Photos: training (5 folds) vs locked test, per place:
side       test  train   All
source                      
Ammariyah   101    913  1014
Madinah      76    409   485
Thumamah     93    934  1027
All         270   2256  2526

Training photos per fold:
fold         0    1    2    3    4
source                            
Ammariyah  104  272  201  186  150
Madinah     94   77   83   84   71
Thumamah   169  219  169  176  201


In [ ]:
# ===== CELL 6: QASSIM (training only): the SAME frozen Qassim folds as the final EdgeNeXt model =====
def qassim_leaf(name):
    m = re.match(r"^QD[^0-9]*?(\d+)[._\- ](\d+)", str(name), flags=re.I)
    return m.group(1) if m else str(name)


mad_run3 = old[~old.name.str.upper().str.startswith("QD")].reset_index(drop=True)
q = old[old.name.str.upper().str.startswith("QD")].copy()
q["cls"] = q.key.map(KEY2NAME)
q["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in q.img]
qb = q.groupby("md5").cls.nunique()
q = q[~q.md5.isin(qb[qb > 1].index)].drop_duplicates("md5").reset_index(drop=True)
mad_md5 = {hashlib.md5(open(p, "rb").read()).hexdigest() for p in mad_run3.img}
q = q[~q.md5.isin(mad_md5)].reset_index(drop=True)
q["group"] = "Qassim_" + q.cls + "_" + q.name.map(qassim_leaf)
q["fold"] = -1
for k, (_, v) in enumerate(grouped_folds(q, N_FOLDS, q.cls)):
    q.loc[v, "fold"] = k
q["fold"] = q.fold.astype(int)
assert (q.groupby("group").fold.nunique() == 1).all(), "LEAK: a Qassim leaf is split across folds"

QFROZEN = (FINAL if os.path.exists(FINAL + "/qassim_split.csv") else SAVE) + "/qassim_split.csv"
qcols = ["name", "cls", "fold"]
os.makedirs(SAVE, exist_ok=True)
if os.path.exists(QFROZEN):
    fz = pd.read_csv(QFROZEN, dtype={"name": str})[qcols]
    if sorted(map(tuple, fz.astype(str).values.tolist())) != sorted(map(tuple, q[qcols].astype(str).values.tolist())):
        raise SystemExit("❌ STOP: the Qassim split CHANGED. Nothing was trained.")
    print("✅ Same split as the frozen one (Qassim):", QFROZEN)
else:
    q[qcols].to_csv(QFROZEN, index=False)
    print("Saved the frozen Qassim split to", QFROZEN)
print("Qassim photos per fold:", q.fold.value_counts().sort_index().to_dict())

✅ Same split as the frozen one (Qassim): /content/drive/MyDrive/Nakhlah_EdgeNeXt_FINAL_model/qassim_split.csv
Qassim photos per fold: {0: 16, 1: 16, 2: 16, 3: 16, 4: 16}


In [ ]:
# ===== CELL 7: build the YOLO dataset: 1 class "leaf", train = folds 1-4, val = fold 0, test = locked 15% =====
YD = "/content/yolo_leaf"
shutil.rmtree(YD, ignore_errors=True)
folder_of = {s: f"/content/ds_{s}" for s in SITES}
folder_of["Madinah"] = "/content/ds_old"
folder_of["Qassim"] = "/content/ds_old"
_names = {}


def names_of(folder):
    if folder not in _names:
        n = yaml.safe_load(open(folder + "/data.yaml"))["names"]
        _names[folder] = n if isinstance(n, list) else [n[k] for k in sorted(n)]
    return _names[folder]


def leaf_lines(img_path, folder):
    """Every box of one of the 4 classes becomes class 0 = leaf (insect boxes are left out)."""
    lab = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    out = []
    for line in open(lab):
        p = line.split()
        if len(p) != 5:
            continue                                   # (polygons are not expected; skipped if present)
        nm = names_of(folder)[int(float(p[0]))]
        if norm(CLASS_MAP.get(nm, nm)) in KEY2NAME:
            out.append("0 " + " ".join(p[1:5]))
    return out


photos = pd.concat([roles.assign(source=roles.source)[["img", "name", "cls", "fold", "source", "side"]],
                    q.assign(source="Qassim", side="train")[["img", "name", "cls", "fold", "source", "side"]]],
                   ignore_index=True)
photos["split"] = np.where(photos.side == "test", "test", np.where(photos.fold == VAL_FOLD, "val", "train"))
photos["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in photos.img]
assert not set(photos[photos.split == "test"].md5) & set(photos[photos.split != "test"].md5), "LEAK: a test photo is also in train/val"
assert (photos[photos.source == "Qassim"].split != "test").all(), "Qassim must never be tested"

nbox = collections.Counter()
for i, r in enumerate(photos.itertuples()):
    lines = leaf_lines(r.img, folder_of[r.source])
    ext = os.path.splitext(r.img)[1].lower()
    os.makedirs(f"{YD}/{r.split}/images", exist_ok=True); os.makedirs(f"{YD}/{r.split}/labels", exist_ok=True)
    stem = f"{r.source}_{i}"
    os.symlink(r.img, f"{YD}/{r.split}/images/{stem}{ext}")
    open(f"{YD}/{r.split}/labels/{stem}.txt", "w").write("\n".join(lines) + ("\n" if lines else ""))
    nbox[r.split] += len(lines)
photos["stem"] = [f"{s}_{i}" for i, s in enumerate(photos.source)]
yaml.safe_dump(dict(path=YD, train="train/images", val="val/images", test="test/images", names={0: "leaf"}),
               open(YD + "/data.yaml", "w"))
print("✅ No leakage: no test photo is in training or validation")
print("\nPhotos per split and place:")
print(pd.crosstab(photos.source, photos.split, margins=True).to_string())
print("\nLeaf boxes per split:", dict(nbox))

✅ No leakage: no test photo is in training or validation

Photos per split and place:
split      test  train  val   All
source                           
Ammariyah   101    809  104  1014
Madinah      76    315   94   485
Qassim        0     64   16    80
Thumamah     93    765  169  1027
All         270   1953  383  2606

Leaf boxes per split: {'train': 4014, 'val': 728, 'test': 517}


In [ ]:
# ===== CELL 8: train YOLO26s (or load it if it was already trained) =====
import torch
from ultralytics import YOLO
RUNS = "/content/runs"
BEST = SAVE + "/nakhlah_yolo26s.pt"
if os.path.exists(BEST):
    print("Already trained: loading", BEST)
else:
    det = YOLO(YOLO_MODEL)
    det.train(data=YD + "/data.yaml", imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=YBATCH,
              seed=SEED, deterministic=True, workers=2, project=RUNS, name="leaf", exist_ok=True, plots=True)
    shutil.copy(f"{RUNS}/leaf/weights/best.pt", BEST)
    for f in ["results.csv", "results.png", "BoxPR_curve.png", "BoxF1_curve.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
        if os.path.exists(f"{RUNS}/leaf/{f}"):
            shutil.copy(f"{RUNS}/leaf/{f}", f"{SAVE}/train_{f}")
    print("✅ Saved best weights (picked on fold 0):", BEST)
det = YOLO(BEST)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_leaf/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fr

In [ ]:
# ===== CELL 9: test the detector ONCE on the locked 15% test (boxes only, no disease) =====
m = det.val(data=YD + "/data.yaml", split="test", imgsz=IMGSZ, batch=YBATCH, conf=0.001, project=RUNS, name="test", exist_ok=True, plots=True)
DET = dict(photos=int((photos.split == "test").sum()), leaf_boxes=nbox["test"],
           precision=float(m.box.mp), recall=float(m.box.mr), mAP50=float(m.box.map50), mAP50_95=float(m.box.map),
           ms_per_photo_colab_T4=float(sum(m.speed.values())))
pd.DataFrame([DET]).to_csv(SAVE + "/detector_test_results.csv", index=False)
for f in ["BoxPR_curve.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    if os.path.exists(f"{RUNS}/test/{f}"):
        shutil.copy(f"{RUNS}/test/{f}", f"{SAVE}/test_{f}")
print("\n========== YOLO26s leaf detector | locked 15% test (used once) ==========")
print(pd.DataFrame([DET]).round(3).to_string(index=False))

Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1770.1±397.1 MB/s, size: 102.0 KB)
val: Scanning /content/yolo_leaf/test/labels... 270 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 270/270 998.8it/s 0.3s
val: New cache created: /content/yolo_leaf/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 2.6it/s 6.6s
                   all        270        517      0.511      0.366      0.379      0.167
Speed: 2.2ms preprocess, 12.7ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /content/runs/test

========== YOLO26s leaf detector | locked 15% test (used once) ==========
 photos  leaf_boxes  precision  recall  mAP50  mAP50_95  ms_per_photo_colab_T4
    270         517      0.511   0.366  0.379     0.167               

In [ ]:
# ===== CELL 10: FULL PIPELINE on the locked test: YOLO26s boxes -> crop (10% pad, 224) -> EdgeNeXt -> photo vote =====
import json, timm
from PIL import Image, ImageOps
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, balanced_accuracy_score
info = json.load(open(FINAL + "/nakhlah_model.json"))
assert info["class_names"] == NAMES, f"class order differs: {info['class_names']} vs {NAMES}"
dev = "cuda" if torch.cuda.is_available() else "cpu"
clf = timm.create_model(info["timm_name"].split(".")[0], pretrained=False, num_classes=info["num_classes"])
clf.load_state_dict(torch.load(FINAL + "/edgenext_nakhlah.pt", map_location="cpu"))
clf = clf.to(dev).eval()
MEAN, STD = np.array(info["mean"], np.float32), np.array(info["std"], np.float32)


def leaf_crops(im, boxes):
    """Same cropping as training: 10% padding, skip boxes under 20 px, resize to 224."""
    W, H = im.size
    out = []
    for x0, y0, x1, y1 in boxes:
        w, h = x1 - x0, y1 - y0
        if w < MIN_PX or h < MIN_PX:
            continue
        c = im.crop((int(max(0, x0 - PAD * w)), int(max(0, y0 - PAD * h)), int(min(W, x1 + PAD * w)), int(min(H, y1 + PAD * h))))
        out.append(np.asarray(c.resize((SIZE, SIZE)), np.float32) / 255.0)
    return out


@torch.no_grad()
def classify(crops):
    x = torch.from_numpy(((np.stack(crops) - MEAN) / STD).transpose(0, 3, 1, 2)).to(dev)
    return torch.softmax(clf(x).float(), 1).argmax(1).cpu().tolist()


test = photos[photos.split == "test"].reset_index(drop=True)
rows = []
for r in test.itertuples():
    im = ImageOps.exif_transpose(Image.open(r.img)).convert("RGB")
    res = det.predict(np.asarray(im)[:, :, ::-1], imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    crops = leaf_crops(im, res.boxes.xyxy.cpu().numpy().tolist())
    preds = classify(crops) if crops else []
    vote = collections.Counter(preds).most_common(1)[0][0] if preds else -1     # -1 = no leaf found
    rows.append(dict(photo=r.name, source=r.source, true=C[r.cls], said=vote, leaves_found=len(crops)))
P = pd.DataFrame(rows)
P.to_csv(SAVE + "/pipeline_test_photos.csv", index=False)
t, p = P.true.tolist(), P.said.tolist()
labels = sorted(set(t))
pr, rc, f1, _ = precision_recall_fscore_support(t, p, labels=labels, zero_division=0)
PIPE = dict(photos=len(P), no_leaf_found=int((P.said == -1).sum()), accuracy=float(np.mean(np.array(t) == np.array(p))),
            balanced_acc=balanced_accuracy_score(t, p), macro_F1=float(f1.mean()))
pd.DataFrame([PIPE]).to_csv(SAVE + "/pipeline_test_results.csv", index=False)
print("--- per class (whole photo) ---")
print(pd.DataFrame(dict(cls=[NAMES[i] for i in labels], precision=pr, recall=rc, F1=f1)).round(3).to_string(index=False))
print(pd.DataFrame(confusion_matrix(t, p, labels=list(range(len(NAMES))) + [-1]),
                   index=["real " + n for n in NAMES] + ["real -"], columns=["said " + n for n in NAMES] + ["no leaf"]).iloc[:len(NAMES)].to_string())
print("\n========== FULL PIPELINE (YOLO26s + EdgeNeXt) | locked test photos (used once) ==========")
print(pd.DataFrame([PIPE]).round(3).to_string(index=False))
print(f"Same pipeline with the HAND-DRAWN boxes: accuracy {GT_PIPELINE['accuracy']:.3f} | macro F1 {GT_PIPELINE['macro_F1']:.3f}")
print("Leaves found per photo: average", round(P.leaves_found.mean(), 2), "| hand-drawn boxes per test photo:", round(nbox["test"] / len(P), 2))

--- per class (whole photo) ---
               cls  precision  recall    F1
    fungalSpotting      0.791   0.500 0.613
  fungalWiltBlight      0.705   0.642 0.672
           healthy      0.871   0.771 0.818
nutrientDeficiency      0.600   0.554 0.576
                         said fungalSpotting  said fungalWiltBlight  said healthy  said nutrientDeficiency  no leaf
real fungalSpotting                       34                      2             3                        5       24
real fungalWiltBlight                      1                     43             3                       15        5
real healthy                               3                      1            54                        4        8
real nutrientDeficiency                    5                     15             2                       36        7

========== FULL PIPELINE (YOLO26s + EdgeNeXt) | locked test photos (used once) ==========
 photos  no_leaf_found  accuracy  balanced_acc  macro_F1
    270             

y_pred contains classes not in y_true


In [ ]:
# ===== CELL 11: files for the Jetson: ONNX copy, settings, and 20 locked-test PHOTOS (5 per class) =====
try:
    onnx = YOLO(BEST).export(format="onnx", imgsz=IMGSZ)
    if os.path.abspath(onnx) != os.path.abspath(SAVE + "/nakhlah_yolo26s.onnx"):
        shutil.copy(onnx, SAVE + "/nakhlah_yolo26s.onnx")
    print("✅ Saved ONNX: nakhlah_yolo26s.onnx")
except Exception as e:
    print("(ONNX export skipped:", str(e)[:200], ")")
json.dump(dict(yolo_model="yolo26s", classes=["leaf"], imgsz=IMGSZ, conf=CONF, crop_padding=PAD, min_box_px=MIN_PX, crop_size=SIZE,
               trained_on="85% of Al Ammariyah + Al Thumamah + Madinah folds 1-4 + Qassim folds 1-4 (fold 0 = validation)",
               detector_test=DET, pipeline_test=PIPE),
          open(SAVE + "/nakhlah_yolo.json", "w"), indent=2, default=float)
PACK = "/content/jetson_test_photos"
shutil.rmtree(PACK, ignore_errors=True); os.makedirs(PACK)
lab = []
for name in NAMES:
    for j, r in enumerate(test[test.cls == name].head(5).itertuples()):
        fn = f"{name}_{j}{os.path.splitext(r.img)[1].lower()}"
        shutil.copy(r.img, f"{PACK}/{fn}")
        lab.append(dict(file=fn, true_class=name))
pd.DataFrame(lab).to_csv(PACK + "/labels.csv", index=False)
shutil.make_archive(SAVE + "/jetson_test_photos", "zip", PACK)
print(f"✅ Jetson test pack: {len(lab)} whole photos -> {SAVE}/jetson_test_photos.zip")
print("\nFiles in", SAVE, ":", sorted(os.listdir(SAVE)))

Ultralytics 8.4.197 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Nakhlah_YOLO26s_leaf_detector/nakhlah_yolo26s.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (19.4 MB)
requirements: Ultralytics requirement ['onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 5 packages in 251ms
Prepared 1 package in 560ms
Installed 1 package in 18ms
 + onnxruntime==1.31.1

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 5.3s, saved as '/content/dr

In [ ]:
# ===== CELL 12: FIX A: never answer "no leaf". Choose the confidence threshold on FOLD 0 (not the test), then test ONCE =====
# Rule: use every box above the threshold; if there is none, use YOLO's single most confident box anyway.
# Note: EdgeNeXt was trained on fold 0 too, so fold-0 scores are optimistic. Fold 0 is only used to CHOOSE the threshold.
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
TOP_K = 20                                         # at most 20 boxes per photo are looked at


def photo_boxes(img_path):
    """All boxes YOLO gives with at least 1% confidence (most confident first) + the EdgeNeXt answer of each box."""
    im = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
    res = det.predict(np.asarray(im)[:, :, ::-1], imgsz=IMGSZ, conf=0.01, max_det=TOP_K, verbose=False)[0]
    xyxy, cf = res.boxes.xyxy.cpu().numpy().tolist(), res.boxes.conf.cpu().numpy().tolist()
    order = np.argsort(cf)[::-1]
    xyxy, cf = [xyxy[i] for i in order], [cf[i] for i in order]
    W, H = im.size
    crops, keep = [], []
    for k, (x0, y0, x1, y1) in enumerate(xyxy):
        w, h = x1 - x0, y1 - y0
        c = im.crop((int(max(0, x0 - PAD * w)), int(max(0, y0 - PAD * h)), int(min(W, x1 + PAD * w)), int(min(H, y1 + PAD * h))))
        crops.append(np.asarray(c.resize((SIZE, SIZE)), np.float32) / 255.0)
        keep.append(w >= MIN_PX and h >= MIN_PX)
    preds = classify(crops) if crops else []
    return [dict(conf=c, pred=p, big=b) for c, p, b in zip(cf, preds, keep)]


def vote(boxes, thr):
    use = [b["pred"] for b in boxes if b["conf"] >= thr and b["big"]]
    if not use and boxes:
        use = [boxes[0]["pred"]]                     # fallback: the most confident box
    return collections.Counter(use).most_common(1)[0][0] if use else -1


def score(rows, thr):
    t = [r["true"] for r in rows]
    p = [vote(r["boxes"], thr) for r in rows]
    _, _, f1, _ = precision_recall_fscore_support(t, p, labels=sorted(set(t)), zero_division=0)
    return dict(threshold=thr, accuracy=float(np.mean(np.array(t) == np.array(p))), balanced_acc=balanced_accuracy_score(t, p),
                macro_F1=float(f1.mean()), no_leaf=int(sum(x == -1 for x in p))), p


val = photos[(photos.split == "val") & (photos.source != "Qassim")].reset_index(drop=True)   # fold 0 of the 3 places
vrows = [dict(true=C[r.cls], boxes=photo_boxes(r.img)) for r in val.itertuples()]
V = pd.DataFrame([score(vrows, t)[0] for t in THRESHOLDS])
print(f"--- choosing the threshold on FOLD 0 ({len(val)} photos of the 3 places) ---")
print(V.round(3).to_string(index=False))
BEST_THR = float(V.sort_values(["macro_F1", "threshold"], ascending=[False, False]).iloc[0].threshold)
print("Chosen threshold (best fold-0 macro F1):", BEST_THR)

# the locked test, ONCE, with the chosen threshold + fallback
trows = [dict(true=C[r.cls], boxes=photo_boxes(r.img)) for r in test.itertuples()]
PIPE_A, p = score(trows, BEST_THR)
t = [r["true"] for r in trows]
pr, rc, f1, _ = precision_recall_fscore_support(t, p, labels=sorted(set(t)), zero_division=0)
print("\n--- per class (whole photo) ---")
print(pd.DataFrame(dict(cls=NAMES, precision=pr, recall=rc, F1=f1)).round(3).to_string(index=False))
print(pd.DataFrame(confusion_matrix(t, p, labels=list(range(len(NAMES)))),
                   index=["real " + n for n in NAMES], columns=["said " + n for n in NAMES]).to_string())
pd.DataFrame([PIPE_A]).to_csv(SAVE + "/pipeline_test_results_fixA.csv", index=False)
print("\n========== FULL PIPELINE + FIX A | locked test photos ==========")
print(pd.DataFrame([dict(setup="YOLO26s, conf 0.25, can say 'no leaf'", **{k: PIPE[k] for k in ["accuracy", "balanced_acc", "macro_F1", "no_leaf_found"]}),
                    dict(setup=f"YOLO26s, conf {BEST_THR} + fallback (fix A)", accuracy=PIPE_A["accuracy"], balanced_acc=PIPE_A["balanced_acc"],
                         macro_F1=PIPE_A["macro_F1"], no_leaf_found=PIPE_A["no_leaf"]),
                    dict(setup="hand-drawn boxes", accuracy=GT_PIPELINE["accuracy"], balanced_acc=np.nan, macro_F1=GT_PIPELINE["macro_F1"], no_leaf_found=0)]
                   ).round(3).to_string(index=False))

j = json.load(open(SAVE + "/nakhlah_yolo.json"))
j.update(conf=BEST_THR, fallback_top_box=True, pipeline_test_fixA=PIPE_A, threshold_search_fold0=V.to_dict("records"))
json.dump(j, open(SAVE + "/nakhlah_yolo.json", "w"), indent=2, default=float)
print("\n✅ nakhlah_yolo.json updated: conf =", BEST_THR, "+ fallback (the Jetson script will read these)")

--- choosing the threshold on FOLD 0 (367 photos of the 3 places) ---
 threshold  accuracy  balanced_acc  macro_F1  no_leaf
      0.05     0.826         0.810     0.812        0
      0.10     0.831         0.823     0.819        0
      0.15     0.828         0.825     0.818        0
      0.20     0.809         0.806     0.798        0
      0.25     0.807         0.800     0.795        0
      0.30     0.807         0.802     0.796        0
      0.40     0.804         0.799     0.792        0
Chosen threshold (best fold-0 macro F1): 0.1

--- per class (whole photo) ---
               cls  precision  recall    F1
    fungalSpotting      0.814   0.706 0.756
  fungalWiltBlight      0.671   0.701 0.686
           healthy      0.776   0.843 0.808
nutrientDeficiency      0.600   0.600 0.600
                         said fungalSpotting  said fungalWiltBlight  said healthy  said nutrientDeficiency
real fungalSpotting                       48                      6             8            